# Wordle: Gemma + LoRA + ES

Train **LoRA adapters + a linear head** on `google/gemma-3-1b-it` with **Evolution Strategies**, supervised warm-start optional. `RUN_PROFILE='smoke'` runs a fast distilgpt2 path for validation; `'gemma_full'` is the production curriculum.

**Requires:** `torch`, `transformers>=4.50.0`, `jinja2>=3.1.0`, `peft`, `numpy`, `matplotlib`. First run downloads HF weights.

## 0. GPU environment setup

Run once on a fresh GPU host. Installs `peft` and ensures `jinja2>=3.1.0`, then prints the visible CUDA device. Set `HF_TOKEN` in the shell before launching the notebook (Gemma is gated). Skip if the kernel is already configured.

In [ ]:
import os, sys, subprocess

def _pip_install(*pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *pkgs]
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)

try:
    import peft  # noqa: F401
except ImportError:
    _pip_install('peft')

try:
    import jinja2
    _v = tuple(int(x) for x in jinja2.__version__.split('.')[:2])
    if _v < (3, 1):
        _pip_install("jinja2>=3.1.0")
except ImportError:
    _pip_install("jinja2>=3.1.0")

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no CUDA device visible — ES + LoRA will be unusably slow on CPU.')

## 1. Environment and imports

Brings `src/` onto `sys.path`, force-reloads `es_wordle` (Jupyter caches imports), and sets up the device + chat-template helpers. If you see `apply_chat_template requires jinja2>=3.1.0`, restart the kernel after installing/upgrading jinja2.

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import torch

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

_here = Path.cwd().resolve()
ROOT = _here.parent if _here.name == "notebooks" else _here
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wordle_env import load_wordle_environment
from wordle_gpt2_policy import WordleGPT2Policy, TRANSFORMERS_AVAILABLE

import es_wordle
importlib.reload(es_wordle)
from es_wordle import train_es_wordle, train_curriculum

if not TRANSFORMERS_AVAILABLE:
    raise ImportError("Install transformers: pip install transformers")


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def default_model_load_kwargs(device: torch.device) -> tuple[dict, str]:
    if device.type != "cuda":
        return {}, "float32"
    if torch.cuda.is_bf16_supported():
        return {"torch_dtype": torch.bfloat16}, "bfloat16"
    return {"torch_dtype": torch.float16}, "float16"


def _parse_version_tuple(version: str) -> tuple[int, ...]:
    parts = []
    for chunk in version.split("."):
        digits = ""
        for ch in chunk:
            if ch.isdigit():
                digits += ch
            else:
                break
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)


def require_chat_template_support() -> None:
    try:
        import jinja2
    except ImportError as exc:
        raise ImportError(
            "Chat-template models require `jinja2>=3.1.0`. Install with `python -m pip install -U 'jinja2>=3.1.0'`, then restart the kernel."
        ) from exc
    installed = getattr(jinja2, "__version__", "0")
    if _parse_version_tuple(installed) < (3, 1, 0):
        raise ImportError(
            f"Chat-template models require `jinja2>=3.1.0`, found {installed}. Upgrade and restart the kernel."
        )


print("ROOT:", ROOT)
print("es_wordle:", es_wordle.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])
if torch.cuda.is_available():
    print("cuda_device:", torch.cuda.get_device_name(0))

## 2. Hyperparameters

Two run profiles:

- **`RUN_PROFILE="smoke"`** — fast distilgpt2 validation path.
- **`RUN_PROFILE="gemma_full"`** — production Gemma + LoRA curriculum.

Notes:
- `MODEL_LOAD_KWARGS` picks `bfloat16` (or `float16` if bf16 isn't supported) on CUDA, `float32` on CPU.
- `vocab_schedule` is the per-stage *secret pool* size; the action space stays fixed at the full guess vocabulary.
- `ALPHA = None` triggers auto-calibration in section 3 from the initial gradient norm.

In [ ]:
import csv

# === Top-level switches =====================================================
SEED = 42
RUN_PROFILE = "gemma_full"  # "smoke" or "gemma_full"
DEVICE = choose_device()

MOCK_ENV = False
USE_PRIME_TARGETS = True
USE_LORA = True
ACTION_GRANULARITY = "char"  # "char" = autoregressive 5-letter generation w/ trie mask; "word" = legacy single-softmax head
LORA_R = 8
RICHER_PROMPT = True

# === ES + warm-start hyperparameters =======================================
WARM_START_LR = 3e-4
SIGMA = 0.02
# Set ALPHA=None to let the calibration probe in §3 pick a step ≈ 0.13 from the initial ‖ĝ‖.
ALPHA = 4.59e-06
NORMALIZE_GRADIENT = False
RANK_FITNESS = True
BASELINE_SUBTRACT = True  # PGPE-lite raw centered fitness; takes precedence over RANK_FITNESS when True
ANTITHETIC = True
COMMON_RANDOM_NUMBERS = True
EMA_BETA = 0.0  # disable momentum; re-enable only if cos(ĝ) is reliably > ~0.1
EVAL_DETERMINISTIC = True
FITNESS_OBJECTIVE = "win_plus_return"
WIN_FITNESS_SCALE = 8.0

# === Profile config =========================================================
PROFILE_CONFIGS = {
    "smoke": {
        "model_name": "distilgpt2",
        "use_chat_template": False,
        "chat_generation_prompt": False,
        "max_prompt_length": 256,
        "vocab_schedule": [8],
        "n_eval_episodes": 1,
        "eval_n_episodes": 4,
        "eval_every": 1,
        "warm_start_steps": 12,
        "n_pop": 4,
        "n_iterations": 2,
        "num_train_examples": 128,
        "num_eval_examples": 16,
    },
    "gemma_full": {
        "model_name": "google/gemma-3-1b-it",
        "use_chat_template": True,
        "chat_generation_prompt": True,
        "max_prompt_length": 512,
        "vocab_schedule": None,  # built below from the 2315-answer pool
        "n_eval_episodes": 16,
        "eval_n_episodes": 50,
        "eval_every": 1,
        "warm_start_steps": 200,
        "n_pop": 64,
        "n_iterations": 200,
        "num_train_examples": 2000,
        "num_eval_examples": 20,
    },
}

if RUN_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"RUN_PROFILE must be one of {sorted(PROFILE_CONFIGS)}, got {RUN_PROFILE!r}")

cfg = PROFILE_CONFIGS[RUN_PROFILE]
MODEL_NAME = cfg["model_name"]
USE_CHAT_TEMPLATE = cfg["use_chat_template"]
if USE_CHAT_TEMPLATE:
    require_chat_template_support()
CHAT_GENERATION_PROMPT = cfg["chat_generation_prompt"]
MAX_PROMPT_LENGTH = cfg["max_prompt_length"]
N_EVAL_EPISODES = cfg["n_eval_episodes"]
EVAL_N_EPISODES = cfg["eval_n_episodes"]
EVAL_EVERY = cfg["eval_every"]
WARM_START_STEPS = cfg["warm_start_steps"]
N_POP = cfg["n_pop"]
N_ITERATIONS = cfg["n_iterations"]
NUM_TRAIN_EXAMPLES = cfg["num_train_examples"]
NUM_EVAL_EXAMPLES = cfg["num_eval_examples"]

# === Vocab wiring ===========================================================
# Secrets = Wordle answer list (2315). Guesses = union CSV + any answers missing
# from it. Secret words are placed first in the ordered guess list so the
# curriculum's prefix-slicing targets the secret pool.
_answers_path = ROOT / "data" / "wordle_answers.txt"
_union_path = ROOT / "data" / "final_5_letter_words_union.csv"

with _answers_path.open("r", encoding="utf-8") as _f:
    _answers_raw = []
    for _line in _f:
        _w = _line.strip()
        if not _w or _w.startswith("#"):
            continue
        _w = _w.upper()
        if len(_w) != 5 or not _w.isalpha():
            continue
        _answers_raw.append(_w)

_guess_raw = []
with _union_path.open("r", encoding="utf-8", newline="") as _f:
    _reader = csv.reader(_f)
    for row in _reader:
        if not row:
            continue
        w = row[0].strip().upper()
        if not w or w == "WORD":
            continue
        if len(w) == 5 and w.isalpha():
            _guess_raw.append(w)

_answers_set = set(_answers_raw)
_guess_set = set(_guess_raw)

SECRET_WORDS = sorted(_answers_set.intersection(_guess_set))
_missing_answers = sorted([w for w in _answers_set if w not in _guess_set])
GUESS_ONLY_WORDS = sorted([w for w in _guess_set if w not in set(SECRET_WORDS)])
ORDERED_GUESS_WORDS = list(SECRET_WORDS) + GUESS_ONLY_WORDS + _missing_answers
ORDERED_GUESS_WORDS = list(dict.fromkeys(ORDERED_GUESS_WORDS))

if not set(SECRET_WORDS).issubset(set(ORDERED_GUESS_WORDS)):
    raise RuntimeError("Secret words must all be present in guess vocabulary.")

# === Secret-pool schedule ===================================================
if MOCK_ENV:
    from wordle_env import MOCK_WORDLE_TARGETS as _MOCK_T
    VOCAB_SCHEDULE = [len(_MOCK_T)]
else:
    VOCAB_SCHEDULE = [len(SECRET_WORDS)]  # curriculum disabled: one stage over the full answer pool

if not VOCAB_SCHEDULE:
    raise ValueError("vocab_schedule must contain at least one stage size.")
if any(b < a for a, b in zip(VOCAB_SCHEDULE, VOCAB_SCHEDULE[1:])):
    raise ValueError(f"vocab_schedule must be non-decreasing, got {VOCAB_SCHEDULE!r}")

INITIAL_VOCAB = VOCAB_SCHEDULE[0]
MAX_VOCAB = len(ORDERED_GUESS_WORDS)
MODEL_LOAD_KWARGS, MODEL_DTYPE_NAME = default_model_load_kwargs(DEVICE)

# === Policy head resize helper =============================================
def patch_policy_guess_vocab(policy, ordered_guess_words):
    """Resize policy.head to match the full guess vocabulary and refresh the trie."""
    new_words = list(ordered_guess_words)
    new_n = len(new_words)

    old_head = policy.head
    in_features = old_head.weight.shape[1]
    device = old_head.weight.device
    dtype = old_head.weight.dtype
    new_head = torch.nn.Linear(in_features, new_n).to(device=device, dtype=dtype)
    torch.nn.init.orthogonal_(new_head.weight, gain=0.01)
    torch.nn.init.zeros_(new_head.bias)
    policy.head = new_head

    policy.words = new_words
    policy.word_to_idx = {w: i for i, w in enumerate(policy.words)}
    policy.idx_to_word = {i: w for i, w in enumerate(policy.words)}
    policy.action_dim = new_n
    if getattr(policy, "action_granularity", "word") == "char":
        # In char mode the head is unused at inference; freeze it so ES doesn't waste
        # perturbation dimensions on a parameter that never affects rollouts.
        policy.head.weight.requires_grad = False
        policy.head.bias.requires_grad = False
        trie_cls = policy.vocab_trie.__class__ if getattr(policy, "vocab_trie", None) is not None else None
        if trie_cls is None:
            from wordle_gpt2_policy import _WordleVocabTrie as trie_cls
        policy.vocab_trie = trie_cls(policy.words)
    return policy

# === Holdout mode + warm-start budget ======================================
# HOLDOUT_MODE controls the train/eval split on the secret pool:
#   "episode" — same vocab in train and eval; held-out signal is fresh episodes.
#   "word"    — disjoint vocab suffix held out for explicit OOV experiments.
# MASKED_EVAL_SANITY_PROBE adds a per-stage greedy eval restricted to in-vocab indices.
HOLDOUT_MODE = "episode"
SECRET_HOLDOUT_FRAC = 0.2  # only used when HOLDOUT_MODE == "word"
MASKED_EVAL_SANITY_PROBE = True

SECRET_POOL_SIZES = list(VOCAB_SCHEDULE)
if HOLDOUT_MODE == "word":
    EVAL_POOL_SIZES = [
        max(2, int(round(v * SECRET_HOLDOUT_FRAC))) if SECRET_HOLDOUT_FRAC > 0 else 0
        for v in SECRET_POOL_SIZES
    ]
    WS_POOL_SIZES = [v - e for v, e in zip(SECRET_POOL_SIZES, EVAL_POOL_SIZES)]
else:
    EVAL_POOL_SIZES = [0 for _ in SECRET_POOL_SIZES]
    WS_POOL_SIZES = list(SECRET_POOL_SIZES)

# Per-stage warm-start budget scales with the training secret-pool size, floored
# at WARM_START_STEPS and capped at WARM_START_STEPS * MAX_WS_SCALE.
MAX_WS_SCALE = 8
WARM_START_STEPS_PER_STAGE = [
    min(
        WARM_START_STEPS * MAX_WS_SCALE,
        max(WARM_START_STEPS, int(round(WARM_START_STEPS * ws / WS_POOL_SIZES[0]))),
    )
    for ws in WS_POOL_SIZES
]
WARM_START_FEEDBACK_CONSISTENT = True

# Per-iter mini-batch over secrets (None disables; an int k restricts each ES iter to k secrets).
PER_ITER_SECRET_SUBSET_SIZE = None
TRACK_BEST_ITER = True
RESTORE_BEST_ON_FINISH = False

# === RNG seeding ============================================================
import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = DEVICE

# === Summary print ==========================================================
if isinstance(N_ITERATIONS, int):
    _iters_desc = f"N_ITERATIONS={N_ITERATIONS} per stage (total={N_ITERATIONS * len(VOCAB_SCHEDULE)})"
else:
    _iters_desc = f"iters_per_stage={list(N_ITERATIONS)} (total={sum(N_ITERATIONS)})"
print(
    f"Profile: {RUN_PROFILE} | Model: {MODEL_NAME} | chat_template: {USE_CHAT_TEMPLATE} | LoRA: {USE_LORA} (r={LORA_R})\n"
    f"  device={device}  model_dtype={MODEL_DTYPE_NAME}  MAX_PROMPT_LENGTH={MAX_PROMPT_LENGTH}\n"
    f"  action_dim (fixed): {MAX_VOCAB}\n"
    f"  secret pool schedule: {SECRET_POOL_SIZES}  "
    f"(holdout_mode={HOLDOUT_MODE}"
    + (f", holdout_frac={SECRET_HOLDOUT_FRAC}" if HOLDOUT_MODE == "word" else "")
    + f", masked_probe={MASKED_EVAL_SANITY_PROBE})\n"
    f"  per-stage ws / eval pool sizes: {list(zip(WS_POOL_SIZES, EVAL_POOL_SIZES))}\n"
    f"  ES: N_POP={N_POP}  {_iters_desc}  n_eval_episodes={N_EVAL_EPISODES}  "
    f"eval_n_episodes={EVAL_N_EPISODES}\n"
    f"  fitness shaping: "
    + (
        "baseline_subtract (raw centered, no std norm; PGPE-lite)"
        if BASELINE_SUBTRACT
        else ("rank_fitness" if RANK_FITNESS else "z-score")
    )
    + "\n"
    f"  warm-start eps per stage: {WARM_START_STEPS_PER_STAGE} "
    f"(total={sum(WARM_START_STEPS_PER_STAGE)}, feedback_consistent={WARM_START_FEEDBACK_CONSISTENT})"
)
if MODEL_LOAD_KWARGS:
    print("model_load_kwargs:", MODEL_LOAD_KWARGS)
print(
    f"  vocab wiring: secrets={len(SECRET_WORDS)} | guesses={len(ORDERED_GUESS_WORDS)} | "
    f"missing_answers_added={len(_missing_answers)}"
)
print("  secrets subset of guesses:", set(SECRET_WORDS).issubset(set(ORDERED_GUESS_WORDS)))
print("  secret pool sizes:", VOCAB_SCHEDULE)
if ALPHA is None:
    print("  ALPHA: auto-calibrate in §3 (will be set from the in-cell probe).")
else:
    print(f"  ALPHA={ALPHA:.2e} (manual override). Calibration probe in §3 will only print suggestions.")

## 3. Run ES training

Builds the policy + envs, calibrates `ALPHA` from a one-shot gradient-norm probe (when `ALPHA=None`), then runs `train_curriculum` (warm-start CE → ES per stage). `verbose=True` prints **every iteration** as a table:

- **TRAIN** — `Fit` (mean fitness), `ES_win` (mean win rate across the population), `popσ` (fitness spread).
- **EVAL** — held-out `Succ`, mean reward, mean turns. Greedy when `EVAL_DETERMINISTIC=True`.
- **OPT** — `Δθ` (parameter drift from init), `|g|` (raw gradient norm), `|Δ|` (applied step norm), `cos(ĝ)` (similarity to previous gradient).
- **SIG** — `ess`/`wins` (signal counts in the population), `dprobe` (post−pre probe delta), `fb%` (trie fallback rate).

After the run, a per-stage attribution table separates the warm-start contribution (`mem_gap`) from the ES contribution (`es_gain`).

In [ ]:
import gc
import random

from wordle_gpt2_warmstart import supervised_warm_start_wordle

if not USE_LORA:
    raise ValueError("This run expects USE_LORA=True in §2 hyperparameters.")

# === Build policy + environments ===========================================
if "policy" in globals():
    del policy
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

run_seed = int(SEED)
random.seed(run_seed)
np.random.seed(run_seed)
torch.manual_seed(run_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(run_seed)

print(f"\n=== LoRA rank {LORA_R} (seed={run_seed}) ===")

policy = WordleGPT2Policy(
    model_name=MODEL_NAME,
    use_prime_targets=USE_PRIME_TARGETS,
    max_vocab_size=MAX_VOCAB,
    max_prompt_length=MAX_PROMPT_LENGTH,
    include_mock_targets_in_vocab=True,
    richer_prompt=RICHER_PROMPT,
    use_chat_template=USE_CHAT_TEMPLATE,
    chat_generation_prompt=CHAT_GENERATION_PROMPT,
    use_lora=USE_LORA,
    action_granularity=ACTION_GRANULARITY,
    lora_r=LORA_R,
    model_kwargs=MODEL_LOAD_KWARGS,
).to(device)
policy = patch_policy_guess_vocab(policy, ORDERED_GUESS_WORDS)

from wordle_env import MOCK_WORDLE_TARGETS
assert all(w in policy.words for w in MOCK_WORDLE_TARGETS), (
    "MOCK_WORDLE_TARGETS missing from policy.words; ES would target words the policy cannot emit."
)
print(f"Trainable (ES): {policy.count_trainable_parameters():,}")
print(f"Total params:   {policy.count_parameters():,}")
print(f"Action dim:     {policy.action_dim} (fixed across curriculum)")
print(f"Chat template:  {policy.use_chat_template}")
print(f"Load kwargs:    {MODEL_LOAD_KWARGS or {'torch_dtype': 'float32'}}")

# Two envs: env_train (warm-start + ES rollouts) and env_eval (post-WS probe + periodic eval).
# train_curriculum re-installs each stage's secret pool on both via set_target_pool.
env_train = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
    target_pool=policy.words,
)
env_eval = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
    target_pool=policy.words,
)
env = env_train  # back-compat alias for ad-hoc inspection

# === ALPHA calibration probe ===============================================
# When ALPHA is None: pick the learning rate that yields initial step ≈ 0.13
# from the measured raw ‖ĝ‖. When ALPHA is a float: just print the suggested
# value for comparison. RNG state is snapshotted/restored so the probe does
# not perturb the training stream.
from es_wordle import es_gradient_estimate_wordle

_target_step = 0.13
_min_step = 0.05

_stage1_pool = list(policy.words[:SECRET_POOL_SIZES[0]])
if HOLDOUT_MODE == "word" and SECRET_HOLDOUT_FRAC > 0 and EVAL_POOL_SIZES[0] > 0:
    _stage1_ws_pool = _stage1_pool[:-EVAL_POOL_SIZES[0]]
else:
    _stage1_ws_pool = _stage1_pool
env_train.set_target_pool(_stage1_ws_pool)

_py_state = random.getstate()
_np_state = np.random.get_state()
_torch_state = torch.get_rng_state()
_cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
_grad = _avg_fit = _fits = _avg_win = _pop_std = _es_wrs = None
try:
    _grad, _avg_fit, _fits, _avg_win, _pop_std, _es_wrs = es_gradient_estimate_wordle(
        policy,
        env_train,
        N=N_POP,
        sigma=SIGMA,
        n_eval_episodes=N_EVAL_EPISODES,
        max_turns=6,
        rank_fitness=RANK_FITNESS,
        fitness_objective=FITNESS_OBJECTIVE,
        win_fitness_scale=WIN_FITNESS_SCALE,
        antithetic=ANTITHETIC,
        common_random_numbers=COMMON_RANDOM_NUMBERS,
        baseline_subtract=BASELINE_SUBTRACT,
    )
    _g_norm = float(_grad.norm().item())
    _ess_rank = len({round(float(f), 6) for f in _fits})
    _win_count = sum(1 for wr in _es_wrs if wr > 0.0)
    _suggested_alpha = float(_target_step / _g_norm) if _g_norm > 1e-8 else None
    if ALPHA is None:
        if _suggested_alpha is None:
            raise RuntimeError(
                "ALPHA auto-calibration failed: probe ‖ĝ‖ is ~0, indicating no ES signal at all."
            )
        ALPHA = _suggested_alpha
        _implied_step = ALPHA * _g_norm
        _alpha_action = f"  AUTO ALPHA = {ALPHA:.2e}  (chosen to give step ≈ {_target_step})"
    else:
        _implied_step = ALPHA * _g_norm
        _alpha_action = f"  manual ALPHA = {ALPHA:.2e}  (set in §2; calibrator only suggesting)"
    print(
        f"\n[ALPHA calibration @ init, action_dim={policy.action_dim}, "
        f"ws_pool={len(_stage1_ws_pool)}, "
        f"trainable={policy.count_trainable_parameters():,}]\n"
        f"  raw ‖ĝ‖           = {_g_norm:.4g}\n"
        f"  ALPHA * ‖ĝ‖       = {_implied_step:.4g}   (target ≈ {_target_step})\n"
        f"  ES probe avg_fit  = {_avg_fit:+.3f}   ES_win = {_avg_win:.1%}   popσ = {_pop_std:.4f}\n"
        f"  ES signal probes  = ess_rank {_ess_rank}/{N_POP}, wins {_win_count}/{N_POP}\n"
        f"{_alpha_action}"
    )
    if _suggested_alpha is not None and _implied_step < _min_step:
        print(
            f"  ⚠ implied step {_implied_step:.4g} < {_min_step} threshold.\n"
            f"    Suggested ALPHA ≈ {_suggested_alpha:.2e} would give step ≈ {_target_step}.\n"
            f"    Set ALPHA=None in §2 to let the calibrator apply this automatically."
        )
finally:
    random.setstate(_py_state)
    np.random.set_state(_np_state)
    torch.set_rng_state(_torch_state)
    if _cuda_state is not None:
        torch.cuda.set_rng_state_all(_cuda_state)
    del _py_state, _np_state, _torch_state, _cuda_state
    del _grad, _avg_fit, _fits, _avg_win, _pop_std, _es_wrs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# === train_curriculum =======================================================
history = train_curriculum(
    policy,
    env_train,
    vocab_schedule=VOCAB_SCHEDULE,
    n_iterations_per_stage=N_ITERATIONS,
    expand_action_space=False,  # action space is fixed at MAX_VOCAB; curriculum varies only the secret pool
    env_eval=env_eval,
    holdout_mode=HOLDOUT_MODE,
    secret_holdout_frac=SECRET_HOLDOUT_FRAC,
    masked_eval_sanity_probe=MASKED_EVAL_SANITY_PROBE,
    warm_start_fn=supervised_warm_start_wordle,
    warm_start_steps=WARM_START_STEPS_PER_STAGE,
    # Once any stage's post-WS in-distribution success exceeds this, suppress warm-start for later stages.
    warm_start_max_post_ws_success=0.85,
    warm_start_kwargs={
        "lr": WARM_START_LR,
        "device": device,
        "seed": run_seed,  # train_curriculum adds 1000 * stage_idx per stage
        "verbose": True,
        "batch_size": 8,
        "feedback_consistent_random": WARM_START_FEEDBACK_CONSISTENT,
    },
    post_warm_start_eval_episodes=EVAL_N_EPISODES,
    post_warm_start_eval_deterministic=EVAL_DETERMINISTIC,
    N=N_POP,
    sigma=SIGMA,
    alpha=ALPHA,
    n_eval_episodes=N_EVAL_EPISODES,
    max_turns=6,
    eval_every=EVAL_EVERY,
    verbose=True,
    normalize_gradient=NORMALIZE_GRADIENT,
    eval_n_episodes=EVAL_N_EPISODES,
    rank_fitness=RANK_FITNESS,
    eval_deterministic=EVAL_DETERMINISTIC,
    fitness_objective=FITNESS_OBJECTIVE,
    win_fitness_scale=WIN_FITNESS_SCALE,
    antithetic=ANTITHETIC,
    common_random_numbers=COMMON_RANDOM_NUMBERS,
    ema_beta=EMA_BETA,
    baseline_subtract=BASELINE_SUBTRACT,
    per_iter_secret_subset_size=PER_ITER_SECRET_SUBSET_SIZE,
    track_best_iter=TRACK_BEST_ITER,
    restore_best_on_finish=RESTORE_BEST_ON_FINISH,
)

final_success = float(history["eval_success"][-1])
print(f"\nFinal success probability (rank={LORA_R}): {final_success:.3f}")

# === Per-stage attribution table ===========================================
# mem_gap = in_dist_post_WS - heldout_post_WS  (warm-start memorization gap)
# es_gain = heldout_end_ES - heldout_post_WS   (ES contribution on top of warm-start)
_holdout_label = list(history.get("stage_holdout_mode", [HOLDOUT_MODE])) or [HOLDOUT_MODE]
_holdout_label = _holdout_label[0]
_ho_col = "fresh_eps" if _holdout_label == "episode" else "ho"
print(
    f"\nPer-stage warm-start vs ES attribution "
    f"(eval_success columns are HELD-OUT, mode={_holdout_label}):"
)
print(
    f"  {'stage':>5} {'ws':>4} {'eval':>4} "
    f"{'in_WS':>7} {_ho_col + '_WS':>7} {_ho_col + '_end':>7} "
    f"{'mask_WS':>8} {'mem_gap':>8} {'es_gain':>8}"
)
_eval_iters = list(history.get("iteration", []))
_eval_succs = list(history.get("eval_success", []))
_stage_starts = list(history.get("stage_starts", []))
_ws_pool_sizes = list(history.get("stage_secret_pool_sizes", []))
_eval_pool_sizes = list(history.get("stage_eval_pool_sizes", []))
_post_ws_indist = list(history.get("post_warmstart_success_indist", []))
_post_ws_heldout = list(history.get("post_warmstart_success_heldout", []))
_post_ws_masked = list(history.get("stage_masked_post_warmstart_success", []))
for s_idx, s_start in enumerate(_stage_starts):
    s_end = _stage_starts[s_idx + 1] - 1 if s_idx + 1 < len(_stage_starts) else (
        _eval_iters[-1] if _eval_iters else s_start
    )
    end_es = float("nan")
    for it, succ in zip(_eval_iters, _eval_succs):
        if s_start <= it <= s_end:
            end_es = float(succ)
    in_ws = _post_ws_indist[s_idx] if s_idx < len(_post_ws_indist) else float("nan")
    ho_ws = _post_ws_heldout[s_idx] if s_idx < len(_post_ws_heldout) else float("nan")
    mask_ws = _post_ws_masked[s_idx] if s_idx < len(_post_ws_masked) else float("nan")
    ws_size = _ws_pool_sizes[s_idx] if s_idx < len(_ws_pool_sizes) else 0
    eval_size = _eval_pool_sizes[s_idx] if s_idx < len(_eval_pool_sizes) else 0
    mem_gap = in_ws - ho_ws if (in_ws == in_ws and ho_ws == ho_ws) else float("nan")
    es_gain = end_es - ho_ws if (end_es == end_es and ho_ws == ho_ws) else float("nan")
    _mask_str = "    n/a " if mask_ws != mask_ws else f"{mask_ws:>+8.1%}"
    print(
        f"  {s_idx + 1:>5} {ws_size:>4} {eval_size:>4} "
        f"{in_ws:>7.1%} {ho_ws:>7.1%} {end_es:>7.1%} "
        f"{_mask_str} {mem_gap:>+8.1%} {es_gain:>+8.1%}"
    )

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

it = np.array(history.get("iteration", []), dtype=float)
ti = np.array(history.get("train_iter", []), dtype=float)

fig, axes = plt.subplots(4, 3, figsize=(16, 12), sharex=False)

# --- Core eval metrics ---
axes[0, 0].plot(it, history.get("eval_success", []), "g-o", ms=3)
axes[0, 0].set_title("Eval success rate")
axes[0, 0].set_xlabel("iteration")
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(it, history.get("eval_reward", []), "b-o", ms=3)
axes[0, 1].set_title("Eval mean reward")
axes[0, 1].set_xlabel("iteration")
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(it, history.get("eval_turns", []), "m-o", ms=3)
axes[0, 2].set_title("Eval mean turns")
axes[0, 2].set_xlabel("iteration")
axes[0, 2].grid(True, alpha=0.3)

# --- Optimization mechanics ---
axes[1, 0].plot(ti, history.get("train_fitness", []), color="c", lw=1)
axes[1, 0].set_title("ES mean fitness (per iter)")
axes[1, 0].set_xlabel("iteration")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(ti, history.get("train_grad_norm", []), color="r", lw=1)
axes[1, 1].set_title("Applied gradient norm")
axes[1, 1].set_xlabel("iteration")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(ti, history.get("param_drift", []), color="k", lw=1)
axes[1, 2].set_title("Parameter drift ‖theta-theta0‖")
axes[1, 2].set_xlabel("iteration")
axes[1, 2].grid(True, alpha=0.3)

# --- Signal quality diagnostics ---
axes[2, 0].plot(ti, history.get("pop_fitness_std", []), color="orange", lw=1)
axes[2, 0].set_title("Population fitness std")
axes[2, 0].set_xlabel("iteration")
axes[2, 0].grid(True, alpha=0.3)

gc_arr = np.array(history.get("train_grad_cos", []), dtype=float)
axes[2, 1].plot(ti, gc_arr, color="purple", lw=1)
axes[2, 1].axhline(0.0, color="gray", lw=0.7, ls=":")
axes[2, 1].set_title("Gradient cosine cos(g_t,g_{t-1})")
axes[2, 1].set_xlabel("iteration")
axes[2, 1].set_ylim(-1.05, 1.05)
axes[2, 1].grid(True, alpha=0.3)

ess_arr = np.array(history.get("train_ess_rank", []), dtype=float)
n_pop_const = int(N_POP) if "N_POP" in globals() else (int(np.nanmax(ess_arr)) if ess_arr.size else 1)
axes[2, 2].plot(ti, ess_arr, color="teal", lw=1)
axes[2, 2].axhline(n_pop_const, color="gray", lw=0.7, ls=":", label=f"N_POP={n_pop_const}")
axes[2, 2].set_title("ESS rank (unique fitness count)")
axes[2, 2].set_xlabel("iteration")
axes[2, 2].set_ylim(0, max(2, n_pop_const + 1))
axes[2, 2].legend(loc="lower right", fontsize=7)
axes[2, 2].grid(True, alpha=0.3)

# --- Probe + trie error indicators ---
probe_delta = np.array(history.get("train_probe_delta", []), dtype=float)
finite_probe = np.isfinite(probe_delta)
axes[3, 0].axhline(0.0, color="gray", lw=0.7, ls=":")
axes[3, 0].plot(ti[finite_probe], probe_delta[finite_probe], "o-", ms=3, color="firebrick", lw=1)
axes[3, 0].set_title("Delta probe success (post-pre)")
axes[3, 0].set_xlabel("iteration")
axes[3, 0].grid(True, alpha=0.3)

fallback_rate = np.array(history.get("train_trie_fallback_rate", []), dtype=float)
axes[3, 1].plot(ti, fallback_rate, color="brown", lw=1)
axes[3, 1].set_title("Trie fallback rate (error indicator)")
axes[3, 1].set_xlabel("iteration")
axes[3, 1].set_ylim(0, 1.05)
axes[3, 1].grid(True, alpha=0.3)

trie_steps = np.array(history.get("train_trie_steps", []), dtype=float)
trie_oov = np.array(history.get("train_trie_oov_words", []), dtype=float)
axes[3, 2].plot(ti, trie_steps, color="slateblue", lw=1, label="trie steps")
axes[3, 2].plot(ti, trie_oov, color="darkred", lw=1, label="oov words")
axes[3, 2].set_title("Trie usage volume")
axes[3, 2].set_xlabel("iteration")
axes[3, 2].legend(loc="upper right", fontsize=7)
axes[3, 2].grid(True, alpha=0.3)

# Curriculum stage boundaries across all plots.
stage_starts = history.get("stage_starts", [])
stage_vocab_sizes = history.get("stage_vocab_sizes", [])
for ax in axes.flat:
    for s_idx, s_iter in enumerate(stage_starts):
        if s_iter <= 0:
            continue
        ax.axvline(s_iter, color="gray", lw=0.8, ls="--", alpha=0.45)
        if ax is axes[0, 0] and s_idx < len(stage_vocab_sizes):
            ax.text(
                s_iter,
                1.02,
                f"N={stage_vocab_sizes[s_idx]}",
                fontsize=7,
                color="gray",
                ha="left",
                va="bottom",
            )

rank_note = f" | LoRA r={LORA_R}" if USE_LORA else ""
schedule_note = f" | curriculum {stage_vocab_sizes}" if len(stage_vocab_sizes) > 1 else ""
plt.suptitle(f"Wordle ES Results Suite | {MODEL_NAME}{rank_note}{schedule_note}")
plt.tight_layout()
plt.show()

## 5. Save checkpoint (optional)

Saves the head, vocab, and history under `models/` when `USE_LORA=False`. With `USE_LORA=True` the cell skips saving to avoid large adapter artifacts.

In [ ]:
if USE_LORA:
    print(f"LoRA rank used for this run: r={LORA_R}")
    print("Skipping checkpoint save for LoRA run to avoid large model artifacts.")
else:
    save_path = ROOT / "models" / f"wordle_gemma_es_head.{RUN_PROFILE}.pt"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "head": policy.head.state_dict(),
        "model_name": MODEL_NAME,
        "words": policy.words,
        "history": history,
        "use_lora": USE_LORA,
        "run_profile": RUN_PROFILE,
        "model_load_kwargs": {k: str(v) for k, v in MODEL_LOAD_KWARGS.items()},
    }
    if getattr(policy, "_lm_trainable", False):
        payload["lm"] = policy.lm.state_dict()
    torch.save(payload, save_path)
    print("Saved:", save_path)